In [3]:
from pathlib import Path
import joblib
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix,
)

model_path = Path(r"XGB_Transactions_Model.pkl")
dataset_path = Path(r"Path\to\ieee-fraud-detection")
threshold = 0.63

model = joblib.load(model_path)

df = pd.read_csv(dataset_path / "train_transaction.csv")

# Match XGB.py: convert both object and string columns before prediction.
for col in df.columns:
    dtype = df[col].dtype
    if (
        pd.api.types.is_object_dtype(dtype)
        or pd.api.types.is_string_dtype(df[col])
    ):
        df[col] = df[col].astype("category")

# XGB.py trains with TransactionID included as a feature.
X = df.drop(columns=["isFraud"])
y = df["isFraud"]

_, X_test, _, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=420,
    stratify=y,
)

y_prob = model.predict_proba(X_test)[:, 1]
y_pred = (y_prob >= threshold).astype(int)


print(classification_report(y_test, y_pred, digits=4, zero_division=0))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

              precision    recall  f1-score   support

           0     0.9910    0.9613    0.9759    113975
           1     0.4158    0.7593    0.5373      4133

    accuracy                         0.9542    118108
   macro avg     0.7034    0.8603    0.7566    118108
weighted avg     0.9709    0.9542    0.9606    118108


Confusion Matrix:
[[109566   4409]
 [   995   3138]]
